In [ ]:
import pandas as pd
import os

## Transfer txt Files to csv files 


In [ ]:
files_txt = "Datatxt"        
files_csv = "Datacsv" 

os.makedirs(files_csv, exist_ok=True)

In [ ]:
for file in os.listdir(files_txt):
    if file.endswith(".txt"):
        file_path = os.path.join(files_txt, file)
        
        df = pd.read_csv(file_path, sep="$", low_memory=False)
        
        new_name = file.replace(".txt", ".csv")
        df.to_csv(os.path.join(files_csv, new_name), index=False)
        
        print(f"Converted: {file}")

## check It's work? 


In [ ]:
demo = pd.read_csv(f"{files_csv}/DEMO25Q2.csv", low_memory=False)

df = demo.copy()
del demo
df.head()

## Function before merge to prevent duplicatis 

In [ ]:
def prepare_table(file_name):
    df_temp = pd.read_csv(f"{files_csv}/{file_name}.csv", low_memory=False)
    
    df_temp = df_temp.groupby("primaryid").agg(
        lambda x: ", ".join(x.astype(str))
    ).reset_index()
    
    df_temp = df_temp.rename(columns=lambda x: x + "_" + file_name.lower() if x != "primaryid" else x)
    
    return df_temp

# Merge All CSVs Files Together 

In [ ]:
import pandas as pd

tables = ["DEMO", "DRUG", "REAC", "THER", "OUTC", "RPSR","INDI"]
all_data = {}

for table in tables:
    dfs = []
    
    for q in ["Q1", "Q2", "Q3", "Q4"]:
        path = f"../Data/Datacsv/{table}25{q}.csv"
        df = pd.read_csv(path, low_memory=False)
        dfs.append(df)
    
    full_df = pd.concat(dfs, ignore_index=True)
    all_data[table] = full_df
    
    print(f"{table} shape:", full_df.shape)

In [ ]:
grouped_data = {}

for name in ["DRUG", "REAC", "THER", "OUTC", "RPSR","INDI"]:
    df = all_data[name]
    
    grouped = df.groupby("primaryid").agg(
        lambda x: ", ".join(x.astype(str))
    ).reset_index()
    
    grouped = grouped.rename(columns=lambda x: x + "_" + name.lower() if x != "primaryid" else x)
    
    grouped_data[name] = grouped
    
    print(f"{name} grouped done")

In [ ]:
final_df = all_data["DEMO"].copy()

for name in grouped_data:
    final_df = final_df.merge(grouped_data[name], on="primaryid", how="left")
    
    print(f"Merged {name}")

# Final Dataset 


In [ ]:
final_df.to_csv("2025_full_data.csv", index=False)